In [0]:
# Export gold_city_hour as JSON files for the daily animation (visualizations/daily-animation).
#
# One file per city plus an index, written to /Volumes/<catalog>/<schema>/<volume>/daily-animation/:
#   cities.json        list of cities (only ref_cities.in_daily_animation), the cities left out with their reason,
#                      export metadata and the description of every exported field
#                      (taken from the column COMMENTs of gold_city_hour, so the page and the table share one semantic layer)
#   <city_id>.json     the city's hours of the last `days` local days (complete days, the last one up to the latest hour)
#
# No business logic here: every value comes from gold_city_hour as it is. Logic belongs in city.sql / calendar.sql.

dbutils.widgets.text("catalog", "dev")
dbutils.widgets.text("schema", "bikeshare")
dbutils.widgets.text("volume", "exports")
dbutils.widgets.text("days", "7")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
volume  = dbutils.widgets.get("volume")
days    = int(dbutils.widgets.get("days"))
table   = f"{catalog}.{schema}.gold_city_hour"
out_dir = f"/Volumes/{catalog}/{schema}/{volume}/daily-animation"
print(table, "->", out_dir, f"(last {days} days)")

In [0]:
# Target folder: a managed volume for exports, created on first run (idempotent)
import os
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume} COMMENT 'Files exported from gold tables (e.g. JSON for the daily animation)'")
os.makedirs(out_dir, exist_ok=True)

In [0]:
# Read the export window: complete local days from (last date - days + 1) up to the latest hour in the table
HOUR_FIELDS = [
    "hour_local", "date_local", "hour_of_day", "weekday_local", "is_weekend",
    "data_status", "is_imputed", "docked_bikes_raw", "docked_bikes",
    "bikes_out_est", "bikes_out_provisional", "bikes_out_share_of_peak",
    "temperature_c", "precipitation_mm", "snowfall_cm", "precipitation_kind", "cloud_cover_pct", "weather_code", "weather_is_forecast",
    "sun_elevation_deg", "light_phase",
]

pdf = spark.sql(f"""
    WITH last_day AS (SELECT MAX(date_local) AS last_date FROM {table})
    SELECT
        g.city_id,
        g.city_name,
        g.n_stations_city,
        g.bikes_out_peak_recent_7d,
        DATE_FORMAT(g.hour_local_ts, "yyyy-MM-dd'T'HH:mm") AS hour_local,
        CAST(g.date_local AS STRING)                       AS date_local,
        g.hour_local                                       AS hour_of_day,
        g.weekday_local, g.is_weekend,
        g.data_status, g.is_imputed, g.docked_bikes_raw, g.docked_bikes,
        g.bikes_out_est, g.bikes_out_provisional, ROUND(g.bikes_out_share_of_peak, 4) AS bikes_out_share_of_peak,
        g.temperature_c, g.precipitation_mm, g.snowfall_cm, g.precipitation_kind, g.cloud_cover_pct, g.weather_code, g.weather_is_forecast,
        g.sun_elevation_deg, g.light_phase,
        DATE_FORMAT(g.sunrise_local, "HH:mm")              AS sunrise_local,
        DATE_FORMAT(g.sunset_local,  "HH:mm")              AS sunset_local,
        ROUND(g.daylight_hours, 2)                         AS daylight_hours
    FROM {table} AS g
    JOIN {catalog}.{schema}.ref_cities AS r                 -- only cities flagged for the animation (rule in reference.sql)
        ON g.city_id = r.city_id AND r.in_daily_animation
    CROSS JOIN last_day AS d
    WHERE g.date_local >= DATE_SUB(d.last_date, {days - 1})
    ORDER BY g.city_id, g.hour_ts
""").toPandas()
print(len(pdf), "rows,", pdf["city_id"].nunique(), "cities,", pdf["hour_local"].min(), "to", pdf["hour_local"].max())

In [0]:
# Field descriptions from the table's column COMMENTs (the semantic layer travels with the data)
import pandas as pd
comments = {r["col_name"]: r["comment"] for r in spark.sql(f"DESCRIBE TABLE {table}").collect() if r["col_name"] and not r["col_name"].startswith("#")}
renamed = {"hour_local": "hour_local_ts", "hour_of_day": "hour_local"}          # export name -> table column
field_docs = {f: comments.get(renamed.get(f, f), "") for f in HOUR_FIELDS}
field_docs["hour_local"]  = "Hour start, Europe/Zurich wall-clock time, yyyy-MM-ddTHH:mm"
field_docs["hour_of_day"] = comments.get("hour_local", "")

# docked_bikes_raw has NULLs, so pandas turns the INT column into float (2001.0); restore integers
pdf["docked_bikes_raw"] = pdf["docked_bikes_raw"].astype("Int64")

def clean(v):
    """JSON-safe values: NaN/NaT/NA -> None, numpy scalars -> Python"""
    if v is None or v is pd.NaT or v is pd.NA or (isinstance(v, float) and v != v):
        return None
    return v.item() if hasattr(v, "item") else v

In [0]:
# Write one file per city, then the index (index last, so a reader never sees an index pointing to a missing file)
import json, datetime

generated_at = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%MZ")
cities = []
for city_id, g in pdf.groupby("city_id", sort=True):
    days_df = g[["date_local", "sunrise_local", "sunset_local", "daylight_hours"]].drop_duplicates("date_local")
    doc = {
        "city_id": city_id,
        "city_name": g["city_name"].iloc[0],
        "generated_at_utc": generated_at,
        "n_stations_city": clean(g["n_stations_city"].max()),
        "bikes_out_peak_recent_7d": clean(g["bikes_out_peak_recent_7d"].max()),
        "days": [{k: clean(v) for k, v in r.items()} for r in days_df.to_dict("records")],
        "hours": [{k: clean(r[k]) for k in HOUR_FIELDS} for r in g.to_dict("records")],
    }
    with open(f"{out_dir}/{city_id}.json", "w", encoding="utf-8") as f:
        json.dump(doc, f, ensure_ascii=False, separators=(",", ":"))
    cities.append({"city_id": city_id, "city_name": doc["city_name"], "n_stations_city": doc["n_stations_city"],
                   "file": f"{city_id}.json", "n_hours": len(doc["hours"])})

index = {
    "generated_at_utc": generated_at,
    "source_table": table,
    "window_days": days,
    "first_hour_local": pdf["hour_local"].min(),
    "last_hour_local": pdf["hour_local"].max(),
    "attribution": "Bike data: sharedmobility.ch (GBFS, Swiss Federal Office of Energy). Weather: Open-Meteo, CC BY 4.0 (MeteoSwiss ICON models).",
    "fields": field_docs,
    "cities": cities,
    "not_shown": [{"city_id": r["city_id"], "city_name": r["city_name"], "note": r["animation_note"]}
                  for r in spark.sql(f"SELECT city_id, city_name, animation_note FROM {catalog}.{schema}.ref_cities WHERE NOT in_daily_animation ORDER BY city_name").collect()],
}
with open(f"{out_dir}/cities.json", "w", encoding="utf-8") as f:
    json.dump(index, f, ensure_ascii=False, indent=1)

print(f"Wrote {len(cities)} city files + cities.json to {out_dir}")
display(pd.DataFrame(cities))